<a href="https://colab.research.google.com/github/zaidlameer/DeetectorPrototype/blob/main/lightWeightAudio_TrainingV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# prompt: mount my drive

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# prompt: visualize the data distribution among the classes available in the /content/drive/MyDrive/audio_files

import os
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming audio files are organized into subdirectories named after the classes
audio_files_dir = "/content/drive/MyDrive/audio_dataset_train"

class_counts = {}
for class_name in os.listdir(audio_files_dir):
    class_path = os.path.join(audio_files_dir, class_name)
    if os.path.isdir(class_path):
        class_counts[class_name] = len([f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))])

# Create a bar plot of the class distribution
classes = list(class_counts.keys())
counts = list(class_counts.values())

plt.figure(figsize=(10, 6))
sns.barplot(x=classes, y=counts)
plt.xlabel("Classes")
plt.ylabel("Number of Audio Files")
plt.title("Data Distribution among Classes")
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability if needed
plt.tight_layout()
plt.show()


In [ ]:
import os
import librosa
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
# --- 1. Data Loading and Preprocessing (Optimized for FLAC) ---

def extract_features(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=None)  # Load audio with original sampling rate (FLAC friendly)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        mfccs_scaled = np.mean(mfccs.T, axis=0)
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None
    return mfccs_scaled

def load_data(real_dir, fake_dir):
    features = []
    labels = []

    for audio_file in os.listdir(real_dir):
        if audio_file.endswith('.flac'): # only process flac files
            feature = extract_features(os.path.join(real_dir, audio_file))
            if feature is not None:
                features.append(feature)
                labels.append(0)  # 0 for real

    for audio_file in os.listdir(fake_dir):
        if audio_file.endswith('.flac'): # only process flac files
            feature = extract_features(os.path.join(fake_dir, audio_file))
            if feature is not None:
                features.append(feature)
                labels.append(1)  # 1 for fake

    return np.array(features), np.array(labels)

In [ ]:
real_audio_dir = "/content/drive/MyDrive/audio_dataset_train/RAFV_audio_test" # Replace with your real audio directory
fake_audio_dir = "/content/drive/MyDrive/audio_dataset_train/FAFV_audio_test"

In [ ]:
features, labels = load_data(real_audio_dir, fake_audio_dir)

X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(40,)),
    tf.keras.layers.Dropout(0.3),  # Add dropout for regularization
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid') # Binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
history = model.fit(X_train, y_train, epochs=50, validation_split=0.2, batch_size=32)

Epoch 1/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.5506 - loss: 12.0785 - val_accuracy: 0.5810 - val_loss: 2.2957
Epoch 2/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6397 - loss: 2.6767 - val_accuracy: 0.6544 - val_loss: 0.5881
Epoch 3/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6409 - loss: 1.0959 - val_accuracy: 0.7278 - val_loss: 0.5607
Epoch 4/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6578 - loss: 0.7026 - val_accuracy: 0.7324 - val_loss: 0.5646
Epoch 5/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6999 - loss: 0.6136 - val_accuracy: 0.7202 - val_loss: 0.5474
Epoch 6/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7175 - loss: 0.5787 - val_accuracy: 0.7569 - val_loss: 0.5115
Epoch 7/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7235 - loss: 0.5623 - val_accuracy: 0.7615 - val_loss: 0.5095
Epoch 8/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7493 - loss: 0.5181 - val_accuracy: 0.7661 - val_los

In [ ]:
y_pred = model.predict(X_test)
y_pred_binary = (y_pred > 0.5).astype(int)

print(classification_report(y_test, y_pred_binary))

26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
              precision    recall  f1-score   support

           0       0.92      0.94      0.93       406
           1       0.94      0.92      0.93       412

    accuracy                           0.93       818
   macro avg       0.93      0.93      0.93       818
weighted avg       0.93      0.93      0.93       818



In [ ]:
model.save('audio_model.keras') # save as keras format
print("Trained model saved as audio_model.keras")


Trained model saved as audio_model.keras


In [ ]:
import tensorflow as tf
import librosa
import numpy as np

def predict_audio(audio_file_path, model_path='audio_detection_model.keras'):
    """
    Predicts whether an audio file is real or fake using a trained Keras model.

    Args:
        audio_file_path: The path to the audio file to predict.
        model_path: The path to the saved Keras model.

    Returns:
        A string indicating "Real" or "Fake", or None if an error occurs.
    """
    try:
        # 1. Load the audio file and extract features
        y, sr = librosa.load(audio_file_path, sr=None)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        mfccs_scaled = np.mean(mfccs.T, axis=0)
        features = np.expand_dims(mfccs_scaled, axis=0)  # Add batch dimension

        # 2. Load the trained model
        model = tf.keras.models.load_model('/content/audio_model.keras')

        # 3. Make the prediction
        prediction = model.predict(features)[0][0]  # Get the single prediction value

        # 4. Interpret the prediction
        if prediction > 0.5:
            return "Fake"
        else:
            return "Real"

    except Exception as e:
        print(f"Error during prediction: {e}")
        return None

# Example usage:
audio_file = '/content/drive/MyDrive/audio_dataset_train/FAFV_audio_test/00001_id00634_z5t9oJizDQg_faceswap_id00495_wavtolip.flac' # Replace with your audio file path
prediction_result = predict_audio(audio_file)

if prediction_result:
    print(f"The audio file is predicted to be: {prediction_result}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
The audio file is predicted to be: Fake


In [ ]:
# --- 8. Model Conversion for Edge Devices (TensorFlow Lite) ---

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('audio_detection_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("TensorFlow Lite model saved as audio_detection_model.tflite")

# --- 9. Optional: Quantization for further size reduction ---

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

with open('audio_detection_quant_model.tflite', 'wb') as f:
    f.write(tflite_quant_model)

print("Quantized TensorFlow Lite model saved as audio_detection_quant_model.tflite")

# --- 10. Optional: Inference example to test the tflite model---
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path="audio_detection_model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

#Example inference with the first test sample
test_sample = np.expand_dims(X_test[0], axis=0).astype(np.float32) #Expand dims and set type.
interpreter.set_tensor(input_details[0]['index'], test_sample)

interpreter.invoke()

output_data = interpreter.get_tensor(output_details[0]['index'])
print("TFLite inference output:", output_data)

Saved artifact at '/tmp/tmp7xkkqa_s'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40), dtype=tf.float32, name='keras_tensor_81')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136917858137296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136917858139216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136917858136336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136917858133840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136917858139984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136917858139408: TensorSpec(shape=(), dtype=tf.resource, name=None)
TensorFlow Lite model saved as audio_detection_model.tflite
Saved artifact at '/tmp/tmp2njqw4f9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40), dtype=tf.float32, name='keras_tensor_81')
Output Type:
  TensorSpec(shape=(None, 1), dty

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('audio_detection_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("TensorFlow Lite model saved as audio_detection_model.tflite")

Saved artifact at '/tmp/tmpieo0ipqv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40), dtype=tf.float32, name='keras_tensor_81')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136917858137296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136917858139216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136917858136336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136917858133840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136917858139984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136917858139408: TensorSpec(shape=(), dtype=tf.resource, name=None)
TensorFlow Lite model saved as audio_detection_model.tflite
